# Aggregate robustness evaluation

This notebook evaluates the adapter checkpoints on a shared clean/noisy test set and saves the results locally.

The evaluation is designed around two choices:

1. Noise is applied only to the task text.
2. Content quality is measured separately from JSON-format compliance.

The main outputs from this notebook are:

- a row-level evaluation file with every model response,
- summary tables with confidence intervals,
- a valid-JSON-only summary table,
- a delta table relative to the 0% training-noise baseline,
- optional plots saved as image files.

The notebook assumes the adapters were already trained and saved locally.

## Local setup

Run this notebook from a local Jupyter environment. The commands below are meant to be run in a terminal before opening the notebook.

### 1. Create and activate a virtual environment

```bash
python -m venv .venv
source .venv/bin/activate
```

### 2. Install packages

```bash
pip install transformers datasets peft bitsandbytes accelerate bert-score evaluate rouge-score openai scikit-learn matplotlib pandas numpy jupyter ipykernel
```

### 3. Set environment variables

At minimum, set a project directory and the Hugging Face token used to access the base model. The OpenAI key is only required if judge-based metrics are enabled.

```bash
export ROBUSTNESS_PROJECT_DIR="$HOME/llama_robustness_project"
export ADAPTER_ROOT="$ROBUSTNESS_PROJECT_DIR"
export HF_HOME="$ROBUSTNESS_PROJECT_DIR/.cache/huggingface"
export HF_TOKEN="your_huggingface_token"
export OPENAI_API_KEY="your_openai_key"   # optional if ENABLE_LLM_JUDGE=0
export MODEL_ID="meta-llama/Llama-3.2-3B-Instruct"
export ENABLE_LLM_JUDGE="1"
export JUDGE_MODEL="gpt-4o-mini"
```

### 4. Expected local file layout

The notebook looks for these files and folders:

```text
$ROBUSTNESS_PROJECT_DIR/
├── llama3-3b-adapter-noise-0/
├── llama3-3b-adapter-noise-25/
├── llama3-3b-adapter-noise-50/
├── llama3-3b-adapter-noise-75/
├── training_variants_metadata.csv                 # optional
└── strict_eval_set_tasknoise_answer_schema.csv    # optional; created automatically if missing
```

If the adapters are stored elsewhere, set `ADAPTER_ROOT` to that directory instead.

### 5. Start Jupyter and run the notebook

```bash
jupyter lab
```

This notebook expects an NVIDIA CUDA environment because the base model is loaded in 4-bit mode for batched evaluation.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# -------------------------------
# Local paths and runtime settings
# -------------------------------
model_id = os.environ.get("MODEL_ID", "meta-llama/Llama-3.2-3B-Instruct")

project_dir = Path(
    os.environ.get("ROBUSTNESS_PROJECT_DIR", "./llama_robustness_project")
).expanduser().resolve()
adapter_root = Path(
    os.environ.get("ADAPTER_ROOT", str(project_dir))
).expanduser().resolve()
hf_home = Path(
    os.environ.get("HF_HOME", str(project_dir / ".cache" / "huggingface"))
).expanduser().resolve()
figures_dir = project_dir / "figures"

project_dir.mkdir(parents=True, exist_ok=True)
adapter_root.mkdir(parents=True, exist_ok=True)
hf_home.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_home)
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

strict_eval_path = project_dir / "strict_eval_set_tasknoise_answer_schema.csv"
raw_eval_path = project_dir / "all_adapters_raw_eval_taskgrounded_batch32.csv"
summary_path = project_dir / "all_adapters_summary_with_ci_taskgrounded_batch32.csv"
conditioned_summary_path = project_dir / "all_adapters_summary_valid_json_only_taskgrounded_batch32.csv"
delta_table_path = project_dir / "all_adapters_delta_vs_baseline_taskgrounded_batch32.csv"
validation_merge_path = project_dir / "all_adapters_summary_with_validation_loss_taskgrounded_batch32.csv"
training_metadata_path = project_dir / "training_variants_metadata.csv"

# -------------------------------
# Reproducibility / runtime knobs
# -------------------------------
GLOBAL_SEED = 42
BOOTSTRAP_SAMPLES = 1000
STRICT_EVAL_SIZE = 200
INFERENCE_BATCH_SIZE = 32
MAX_INPUT_LENGTH = 768
MAX_NEW_TOKENS = 180
TEMPERATURE = 0.0

ENABLE_LLM_JUDGE = os.environ.get("ENABLE_LLM_JUDGE", "1").strip().lower() not in {"0", "false", "no"}
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4o-mini")

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

print(f"Project directory: {project_dir}")
print(f"Adapter root: {adapter_root}")
print(f"Hugging Face cache: {hf_home}")
print(f"Judge metrics enabled: {ENABLE_LLM_JUDGE}")
print(f"Configured batch size: {INFERENCE_BATCH_SIZE}")

## Load the tokenizer and define the adapter-loading helper

Each adapter is evaluated on a fresh copy of the same base model so that the runs remain comparable.

This section also sets left padding for batched generation. That matters for decoder-only models, because the generated tokens need to be sliced from the same padded input width for every example in the batch.

In [ ]:
import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook expects an NVIDIA CUDA environment because the model is loaded in 4-bit mode."
    )

device = "cuda"
print(f"Using device: {device}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    token=os.environ.get("HF_TOKEN"),
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer padding_side: {tokenizer.padding_side}")
print(f"pad_token_id: {tokenizer.pad_token_id}, eos_token_id: {tokenizer.eos_token_id}")

def load_model_with_adapter(adapter_path: str):
    print(f"Loading base model with adapter: {adapter_path}")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model

## Define the evaluation metrics

The generated output is first checked for valid JSON and for the required keys. After that, the notebook extracts the `answer` field and compares it with the gold reference.

This keeps the content metrics focused on task quality instead of letting formatting errors dominate every score. The judge-based metrics are optional and can be disabled with `ENABLE_LLM_JUDGE=0`.

In [ ]:

import json
import re
from typing import Any, Dict, Optional, Tuple
from evaluate import load
from openai import OpenAI

bertscore = load("bertscore")
rouge = load("rouge")

def load_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key and ENABLE_LLM_JUDGE:
        raise ValueError(
            "Set OPENAI_API_KEY in your local environment before running judge-based metrics, "
            "or disable them with ENABLE_LLM_JUDGE=0."
        )
    return OpenAI(api_key=api_key) if api_key else None

client = load_openai_client()

def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_json_object(raw_text: str) -> Tuple[Optional[dict], str]:
    if raw_text is None:
        return None, ""

    text = str(raw_text).strip()
    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0].strip()

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
    else:
        candidate = text

    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return parsed, candidate
        return None, candidate
    except Exception:
        return None, candidate

def get_json_metrics(output: str, required_keys=("answer", "summary", "components")) -> Dict[str, Any]:
    parsed, candidate = extract_json_object(output)
    if parsed is None:
        return {
            "json_score": 0.0,
            "valid_json": 0.0,
            "parsed_json": None,
            "missing_keys": list(required_keys),
        }

    missing_keys = [k for k in required_keys if k not in parsed]
    if len(missing_keys) == 0:
        json_score = 1.0
        valid_json = 1.0
    else:
        json_score = 0.5
        valid_json = 0.0

    return {
        "json_score": json_score,
        "valid_json": valid_json,
        "parsed_json": parsed,
        "missing_keys": missing_keys,
    }

def extract_answer_text(output: str) -> str:
    json_metrics = get_json_metrics(output)
    parsed = json_metrics["parsed_json"]
    if parsed is None:
        return normalize_text(output)

    answer = parsed.get("answer", "")
    summary = parsed.get("summary", "")
    components = parsed.get("components", [])

    pieces = []
    if isinstance(answer, str) and normalize_text(answer):
        pieces.append(normalize_text(answer))
    if isinstance(summary, str) and normalize_text(summary) and normalize_text(summary) not in pieces:
        pieces.append(normalize_text(summary))
    if isinstance(components, list):
        component_text = "; ".join([normalize_text(x) for x in components if normalize_text(x)])
        if component_text:
            pieces.append(component_text)

    fallback = " ".join([p for p in pieces if p]).strip()
    return fallback if fallback else normalize_text(output)

def get_semantic_score(answer_text: str, reference: str) -> float:
    answer_text = normalize_text(answer_text)
    reference = normalize_text(reference)
    if not answer_text or not reference:
        return 0.0
    results = bertscore.compute(predictions=[answer_text], references=[reference], lang="en")
    return float(results["f1"][0])

def get_rouge_l_score(answer_text: str, reference: str) -> float:
    answer_text = normalize_text(answer_text)
    reference = normalize_text(reference)
    if not answer_text or not reference:
        return 0.0
    results = rouge.compute(predictions=[answer_text], references=[reference], use_aggregator=False)
    return float(results["rougeL"][0])

def get_dual_judge_scores(prompt: str, answer_text: str, reference: str) -> Dict[str, float]:
    if not ENABLE_LLM_JUDGE or client is None:
        return {"judge_score": np.nan, "reference_judge_score": np.nan}

    system_instruction = """
    You are an expert evaluator.
    Score the candidate answer on two axes from 1.0 to 5.0:
    1. prompt_quality: how well the answer addresses the user prompt.
    2. reference_alignment: how well the answer matches the gold reference answer.

    Output only a valid JSON object with exactly two keys:
    {"prompt_quality": <float>, "reference_alignment": <float>}
    """

    judge_prompt = f"""
    <user_prompt>
    {prompt}
    </user_prompt>

    <candidate_answer>
    {answer_text}
    </candidate_answer>

    <gold_reference_answer>
    {reference}
    </gold_reference_answer>
    """

    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": judge_prompt},
            ],
            temperature=0.0,
        )
        content = response.choices[0].message.content.strip()
        parsed, _ = extract_json_object(content)
        if parsed is None:
            raise ValueError(f"Could not parse judge JSON from: {content}")

        pq = float(parsed.get("prompt_quality", 1.0))
        ra = float(parsed.get("reference_alignment", 1.0))
        pq = max(1.0, min(5.0, pq))
        ra = max(1.0, min(5.0, ra))
        return {
            "judge_score": (pq - 1.0) / 4.0,
            "reference_judge_score": (ra - 1.0) / 4.0,
        }
    except Exception as e:
        print(f"Judge error: {e}")
        return {"judge_score": 0.0, "reference_judge_score": 0.0}

def score_example(judge_prompt: str, output: str, reference: str) -> Dict[str, Any]:
    json_metrics = get_json_metrics(output)
    answer_text = extract_answer_text(output)

    semantic_score = get_semantic_score(answer_text, reference)
    rougeL_score = get_rouge_l_score(answer_text, reference)
    judge_scores = get_dual_judge_scores(judge_prompt, answer_text, reference)

    valid_json = json_metrics["valid_json"]
    conditioned = {
        "semantic_score_valid_json": semantic_score if valid_json == 1.0 else np.nan,
        "judge_score_valid_json": judge_scores["judge_score"] if valid_json == 1.0 else np.nan,
        "reference_judge_score_valid_json": judge_scores["reference_judge_score"] if valid_json == 1.0 else np.nan,
        "rougeL_valid_json": rougeL_score if valid_json == 1.0 else np.nan,
    }

    return {
        "json_score": round(float(json_metrics["json_score"]), 4),
        "valid_json": round(float(valid_json), 4),
        "semantic_score": round(float(semantic_score), 4),
        "judge_score": round(float(judge_scores["judge_score"]), 4) if not np.isnan(judge_scores["judge_score"]) else np.nan,
        "reference_judge_score": round(float(judge_scores["reference_judge_score"]), 4) if not np.isnan(judge_scores["reference_judge_score"]) else np.nan,
        "rougeL": round(float(rougeL_score), 4),
        "extracted_answer": answer_text,
        "missing_keys": ", ".join(json_metrics["missing_keys"]),
        **conditioned,
    }

def bootstrap_mean_ci(values, n_boot=BOOTSTRAP_SAMPLES, ci=95, seed=GLOBAL_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(sample.mean())

    alpha = (100 - ci) / 2
    return (
        float(values.mean()),
        float(np.percentile(boot_means, alpha)),
        float(np.percentile(boot_means, 100 - alpha)),
    )

def summarize_with_ci(eval_df: pd.DataFrame, group_cols, metrics):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    rows = []
    for group_values, group_df in eval_df.groupby(group_cols):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        row = {col: val for col, val in zip(group_cols, group_values)}
        row["n_examples"] = int(len(group_df))
        for metric in metrics:
            mean_val, ci_low, ci_high = bootstrap_mean_ci(group_df[metric].values)
            row[f"{metric}_mean"] = round(mean_val, 4) if not np.isnan(mean_val) else np.nan
            row[f"{metric}_ci_low"] = round(ci_low, 4) if not np.isnan(ci_low) else np.nan
            row[f"{metric}_ci_high"] = round(ci_high, 4) if not np.isnan(ci_high) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

MAIN_METRICS = ["valid_json", "json_score", "semantic_score", "judge_score", "reference_judge_score", "rougeL"]
VALID_JSON_ONLY_METRICS = ["semantic_score_valid_json", "judge_score_valid_json", "reference_judge_score_valid_json", "rougeL_valid_json"]

print("Evaluation metrics loaded.")


## Build or reload the evaluation set

This section creates the clean/noisy evaluation split used for every adapter. The clean and noisy versions share the same underlying task, and the JSON schema instruction is appended after the task text is built.

If a saved evaluation file already exists locally and matches the expected format, the notebook reuses it. Otherwise it rebuilds the file and writes it to the project directory.

In [ ]:

from datasets import load_dataset
from sklearn.model_selection import train_test_split

REQUIRED_EVAL_COLUMNS = {
    "instruction", "input", "output",
    "noisy_instruction", "noisy_input",
    "clean_prompt", "noisy_prompt",
    "clean_judge_prompt", "noisy_judge_prompt",
}

JSON_CONSTRAINT = (
    "Respond ONLY as a valid JSON object with exactly three keys: "
    "'answer' (the full answer to the task as a string), "
    "'summary' (a brief summary string), and "
    "'components' (a list of key concepts)."
)

def normalize_text_for_prompt(text):
    if pd.isna(text):
        return ""
    return str(text).strip()

def typo_noise(text, rng):
    text = normalize_text_for_prompt(text)
    if len(text) < 4:
        return text
    chars = list(text)
    candidate_positions = [i for i in range(len(chars) - 1) if chars[i].isalpha() and chars[i + 1].isalpha()]
    if not candidate_positions:
        return text
    pos = rng.choice(candidate_positions)
    chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
    return "".join(chars)

def punctuation_noise(text, rng):
    text = normalize_text_for_prompt(text)
    return re.sub(r"[.,!?;:]", "", text)

def shorthand_noise(text, rng):
    text = normalize_text_for_prompt(text)
    replacements = {
        "you": "u",
        "are": "r",
        "please": "pls",
        "people": "ppl",
        "with": "w/",
        "without": "w/o",
        "before": "b4",
    }
    words = text.split()
    new_words = [replacements.get(w.lower(), w) for w in words]
    return " ".join(new_words)

def word_drop_noise(text, rng):
    text = normalize_text_for_prompt(text)
    words = text.split()
    if len(words) <= 3:
        return text
    drop_idx = int(rng.integers(0, len(words)))
    words.pop(drop_idx)
    return " ".join(words)

def lowercase_noise(text, rng):
    return normalize_text_for_prompt(text).lower()

def apply_mixed_noise(text, rng):
    text = normalize_text_for_prompt(text)
    if not text:
        return text
    ops = [typo_noise, punctuation_noise, shorthand_noise, word_drop_noise, lowercase_noise]
    n_ops = int(rng.integers(1, 3))
    chosen_ops = rng.choice(ops, size=n_ops, replace=False)
    out = text
    for op in chosen_ops:
        out = op(out, rng)
    return out

def build_task_prompt(instruction, input_text):
    instruction = normalize_text_for_prompt(instruction)
    input_text = normalize_text_for_prompt(input_text)
    if input_text:
        task_text = f"Instruction: {instruction}\nInput: {input_text}"
    else:
        task_text = f"Instruction: {instruction}"
    return task_text

def build_inference_prompt(task_text):
    return f"{task_text}\n\n{JSON_CONSTRAINT}"

def build_judge_prompt(instruction, input_text):
    instruction = normalize_text_for_prompt(instruction)
    input_text = normalize_text_for_prompt(input_text)
    if input_text:
        return f"Instruction: {instruction}\nInput: {input_text}"
    return f"Instruction: {instruction}"

if os.path.exists(strict_eval_path):
    test_df = pd.read_csv(strict_eval_path)
    if REQUIRED_EVAL_COLUMNS.issubset(set(test_df.columns)) and len(test_df) == STRICT_EVAL_SIZE:
        print(f"Loaded existing strict evaluation set from: {strict_eval_path}")
    else:
        print("Existing strict evaluation file is missing expected columns; rebuilding it.")
        test_df = None
else:
    test_df = None

if test_df is None:
    dataset = load_dataset("yahma/alpaca-cleaned", split="train")
    df = dataset.to_pandas()[["instruction", "input", "output"]].copy()
    df["instruction"] = df["instruction"].fillna("").astype(str).str.strip()
    df["input"] = df["input"].fillna("").astype(str).str.strip()
    df["output"] = df["output"].fillna("").astype(str).str.strip()

    train_df, holdout_df = train_test_split(df, test_size=0.10, random_state=GLOBAL_SEED)
    _, test_df = train_test_split(holdout_df, test_size=STRICT_EVAL_SIZE, random_state=GLOBAL_SEED)
    test_df = test_df.reset_index(drop=True).copy()

    noisy_instructions = []
    noisy_inputs = []
    for idx, row in test_df.iterrows():
        rng = np.random.default_rng(GLOBAL_SEED + idx)
        noisy_instructions.append(apply_mixed_noise(row["instruction"], rng))
        noisy_inputs.append(apply_mixed_noise(row["input"], rng) if normalize_text_for_prompt(row["input"]) else "")

    test_df["noisy_instruction"] = noisy_instructions
    test_df["noisy_input"] = noisy_inputs

    test_df["clean_judge_prompt"] = test_df.apply(lambda row: build_judge_prompt(row["instruction"], row["input"]), axis=1)
    test_df["noisy_judge_prompt"] = test_df.apply(lambda row: build_judge_prompt(row["noisy_instruction"], row["noisy_input"]), axis=1)

    test_df["clean_prompt"] = test_df.apply(
        lambda row: build_inference_prompt(build_task_prompt(row["instruction"], row["input"])), axis=1
    )
    test_df["noisy_prompt"] = test_df.apply(
        lambda row: build_inference_prompt(build_task_prompt(row["noisy_instruction"], row["noisy_input"])), axis=1
    )

    test_df.to_csv(strict_eval_path, index=False)
    print(f"Saved evaluation set to: {strict_eval_path}")

print(f"Strict evaluation set size: {len(test_df)}")


## Inspect a few clean/noisy prompt pairs

Before running the full evaluation, it is useful to spot-check a few examples. The goal is to confirm that the corruption is limited to the task text and that the schema instruction is unchanged across the clean and noisy versions.

In [ ]:
preview_cols = [
    "instruction", "input",
    "noisy_instruction", "noisy_input",
    "clean_judge_prompt", "noisy_judge_prompt",
]
display(test_df[preview_cols].sample(n=5, random_state=GLOBAL_SEED))

sample_idx = 0
print("----- CLEAN INFERENCE PROMPT -----")
print(test_df.loc[sample_idx, "clean_prompt"][:1500])
print("\n----- NOISY INFERENCE PROMPT -----")
print(test_df.loc[sample_idx, "noisy_prompt"][:1500])

print(
    "\nJSON constraint identical across clean/noisy:",
    JSON_CONSTRAINT in test_df.loc[sample_idx, "clean_prompt"] and JSON_CONSTRAINT in test_df.loc[sample_idx, "noisy_prompt"],
)


## Run batched generation and scoring

This is the main evaluation loop. For each adapter, the notebook generates responses for both the clean and noisy prompts, scores the outputs, and appends the results to the local summary tables.

Generation starts at batch size 32. If a CUDA out-of-memory error occurs, the loop temporarily reduces the batch size and retries so the run can continue without manual edits.

In [ ]:
from tqdm.auto import tqdm

noise_ratios = [0, 25, 50, 75]
all_summaries = []
all_conditioned_summaries = []
master_eval_df = pd.DataFrame()

def build_inference_jobs(eval_df):
    jobs = []
    for index, row in eval_df.iterrows():
        jobs.append({
            "id": index,
            "condition": "clean",
            "prompt": row["clean_prompt"],
            "judge_prompt": row["clean_judge_prompt"],
            "target_output": row["output"],
        })
        jobs.append({
            "id": index,
            "condition": "noisy",
            "prompt": row["noisy_prompt"],
            "judge_prompt": row["noisy_judge_prompt"],
            "target_output": row["output"],
        })
    return pd.DataFrame(jobs)

def generate_in_batches(model, prompts, batch_size=INFERENCE_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    all_text = []
    idx = 0
    current_batch_size = batch_size

    pbar = tqdm(total=len(prompts), desc="Batched generation")
    while idx < len(prompts):
        take = min(current_batch_size, len(prompts) - idx)
        batch_prompts = prompts[idx: idx + take]

        try:
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            generation_kwargs = dict(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            if TEMPERATURE and TEMPERATURE > 0:
                generation_kwargs["do_sample"] = True
                generation_kwargs["temperature"] = TEMPERATURE
            else:
                generation_kwargs["do_sample"] = False

            with torch.inference_mode():
                outputs = model.generate(**generation_kwargs)

            input_seq_len = inputs["input_ids"].shape[1]
            for row_idx in range(outputs.shape[0]):
                generated_ids = outputs[row_idx][input_seq_len:]
                text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                all_text.append(text)

            idx += take
            pbar.update(take)
            current_batch_size = batch_size

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if current_batch_size == 1:
                raise RuntimeError(
                    "Out of memory even at batch size 1. Reduce MAX_INPUT_LENGTH or MAX_NEW_TOKENS."
                )
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM encountered. Retrying with smaller batch size: {current_batch_size}")

    pbar.close()
    return all_text

jobs_df = build_inference_jobs(test_df)
print(f"Prepared {len(jobs_df)} inference jobs ({len(test_df)} examples × 2 conditions).")

for ratio in noise_ratios:
    adapter_path = adapter_root / f"llama3-3b-adapter-noise-{ratio}"
    if not adapter_path.exists():
        raise FileNotFoundError(f"Adapter directory not found: {adapter_path}")

    print(f"\n{'=' * 50}")
    print(f"Evaluating adapter trained with {ratio}% noise")
    print(f"{'=' * 50}")

    model = load_model_with_adapter(str(adapter_path))

    generated_outputs = generate_in_batches(
        model=model,
        prompts=jobs_df["prompt"].tolist(),
        batch_size=INFERENCE_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    ratio_results_df = jobs_df.copy()
    ratio_results_df["training_noise_ratio"] = ratio
    ratio_results_df["generated_output"] = generated_outputs

    scored_rows = []
    for _, row in tqdm(ratio_results_df.iterrows(), total=len(ratio_results_df), desc=f"Scoring ({ratio}%)"):
        metrics = score_example(
            judge_prompt=row["judge_prompt"],
            output=row["generated_output"],
            reference=row["target_output"],
        )
        row_data = row.to_dict()
        row_data.update(metrics)
        scored_rows.append(row_data)

    eval_df = pd.DataFrame(scored_rows)
    master_eval_df = pd.concat([master_eval_df, eval_df], ignore_index=True)

    summary = summarize_with_ci(eval_df, ["condition"], MAIN_METRICS)
    summary["training_noise_ratio"] = ratio
    all_summaries.append(summary)

    conditioned_summary = summarize_with_ci(eval_df, ["condition"], VALID_JSON_ONLY_METRICS)
    conditioned_summary["training_noise_ratio"] = ratio
    all_conditioned_summaries.append(conditioned_summary)

    print(f"Main summary for the {ratio}% adapter:")
    display(summary)
    print(f"Content metrics conditioned on valid JSON for the {ratio}% adapter:")
    display(conditioned_summary)

    # Save intermediate results so progress is not lost if a later run fails.
    current_master_summary = pd.concat(all_summaries, ignore_index=True)
    current_conditioned_summary = pd.concat(all_conditioned_summaries, ignore_index=True)

    master_eval_df.to_csv(raw_eval_path, index=False)
    current_master_summary.to_csv(summary_path, index=False)
    current_conditioned_summary.to_csv(conditioned_summary_path, index=False)

    del model
    gc.collect()
    torch.cuda.empty_cache()

final_master_summary = pd.concat(all_summaries, ignore_index=True)
final_master_summary = final_master_summary.sort_values(["training_noise_ratio", "condition"]).reset_index(drop=True)

final_conditioned_summary = pd.concat(all_conditioned_summaries, ignore_index=True)
final_conditioned_summary = final_conditioned_summary.sort_values(["training_noise_ratio", "condition"]).reset_index(drop=True)

master_eval_df.to_csv(raw_eval_path, index=False)
final_master_summary.to_csv(summary_path, index=False)
final_conditioned_summary.to_csv(conditioned_summary_path, index=False)

print(f"Saved raw evaluations to: {raw_eval_path}")
print(f"Saved main summary with confidence intervals to: {summary_path}")
print(f"Saved valid-JSON-only summary to: {conditioned_summary_path}")
print("Evaluation complete.")

## Merge validation loss from training metadata

If `training_variants_metadata.csv` is available, this section attaches the best clean validation loss to each adapter summary row.

That extra table is useful when discussing the tradeoff between robustness gains on noisy inputs and clean-distribution performance.

In [ ]:

summary_with_validation = final_master_summary.copy()

if os.path.exists(training_metadata_path):
    training_meta = pd.read_csv(training_metadata_path)
    candidate_cols = [c for c in training_meta.columns]
    display(training_meta.head())

    ratio_col = None
    loss_col = None
    for c in candidate_cols:
        if c.lower() in {"noise_ratio", "training_noise_ratio", "ratio"}:
            ratio_col = c
        if c.lower() in {"best_eval_loss", "best_validation_loss", "eval_loss", "validation_loss"}:
            loss_col = c

    if ratio_col is not None and loss_col is not None:
        merged_loss = training_meta[[ratio_col, loss_col]].drop_duplicates().rename(
            columns={ratio_col: "training_noise_ratio", loss_col: "best_validation_loss"}
        )
        summary_with_validation = summary_with_validation.merge(merged_loss, on="training_noise_ratio", how="left")
        summary_with_validation.to_csv(validation_merge_path, index=False)
        print(f"Saved summary merged with validation loss to: {validation_merge_path}")
        display(summary_with_validation)
    else:
        print("Could not infer noise-ratio and validation-loss columns from training metadata.")
        print("Available columns:", candidate_cols)
else:
    print(f"Training metadata file not found at: {training_metadata_path}")


## Build the delta table relative to the 0% baseline

This table compares each noisy-training variant against the 0% training-noise model within the same evaluation condition.

Positive deltas indicate improvement over the baseline for that metric, while negative deltas indicate a cost relative to the clean-trained adapter.

In [ ]:

key_metrics_for_delta = ["valid_json_mean", "semantic_score_mean", "judge_score_mean", "reference_judge_score_mean", "rougeL_mean"]

baseline = final_master_summary[final_master_summary["training_noise_ratio"] == 0].copy()
baseline = baseline[["condition"] + key_metrics_for_delta].rename(
    columns={m: f"baseline_{m}" for m in key_metrics_for_delta}
)

delta_df = final_master_summary.merge(baseline, on="condition", how="left")
for metric in key_metrics_for_delta:
    delta_df[f"delta_{metric}"] = delta_df[metric] - delta_df[f"baseline_{metric}"]

delta_cols = [
    "training_noise_ratio", "condition",
    *key_metrics_for_delta,
    *[f"delta_{m}" for m in key_metrics_for_delta],
]
delta_df = delta_df[delta_cols].sort_values(["training_noise_ratio", "condition"]).reset_index(drop=True)
delta_df.to_csv(delta_table_path, index=False)

print(f"Saved baseline-delta table to: {delta_table_path}")
display(delta_df)


## Plot the main trends and save the figures locally

The plots below separate formatting success from task quality. Each figure is displayed in the notebook and also written to the local `figures/` directory so the same outputs can be reused in a report or slide deck.

In [ ]:
import matplotlib.pyplot as plt

plot_df = final_master_summary.sort_values(["training_noise_ratio", "condition"])

fig_metrics = [
    ("valid_json_mean", "Valid JSON rate"),
    ("reference_judge_score_mean", "Reference-grounded judge score"),
    ("semantic_score_mean", "Semantic score (BERTScore F1)"),
]

for metric, title in fig_metrics:
    plt.figure(figsize=(9, 5))
    for condition in ["clean", "noisy"]:
        cond_df = plot_df[plot_df["condition"] == condition].sort_values("training_noise_ratio")
        ci_base = metric.replace("_mean", "")
        plt.plot(cond_df["training_noise_ratio"], cond_df[metric], marker="o", label=condition.title())
        plt.fill_between(
            cond_df["training_noise_ratio"],
            cond_df[f"{ci_base}_ci_low"],
            cond_df[f"{ci_base}_ci_high"],
            alpha=0.2,
        )

    plt.title(title)
    plt.xlabel("Training noise ratio (%)")
    plt.ylabel(title)
    plt.grid(True)
    plt.legend()

    figure_path = figures_dir / f"{metric}_trend.png"
    plt.savefig(figure_path, bbox_inches="tight", dpi=200)
    print(f"Saved figure to: {figure_path}")
    plt.show()
    plt.close()

## Final outputs

At this point the notebook has produced the main artifacts needed for analysis:

1. `final_master_summary` for condition-level results,
2. `final_conditioned_summary` for content scores conditioned on valid JSON,
3. `delta_df` for changes relative to the 0% baseline,
4. `summary_with_validation` if training metadata was available.

The last cell prints these tables in the notebook for a final check after the CSV files have been saved.

In [ ]:
print("Main condition-level summary:")
display(final_master_summary)

print("\nContent metrics conditioned on valid JSON:")
display(final_conditioned_summary)

if "summary_with_validation" in globals():
    print("\nSummary merged with clean validation loss (if available):")
    display(summary_with_validation)
